# Climate TRACE emissions, first look

This notebook is my first pass over the Climate TRACE emissions data. I wanted to understand its shape, its grain and how clean it is before designing the bronze and silver layers.

The input is release v5.11.0, downloaded on 29 September 2026 with the custom download builder. I asked for all US sources in every sector, CO2e on the 100 year GWP, monthly values from January 2021 to December 2025, at source level. The files sit in `data/climate_trace/v5.11.0/emissions/`, which Git ignores. Climate TRACE publishes the data under CC BY 4.0.

The CSV is 9.3 GB, far too big to load into pandas, so I used DuckDB 1.5.6 locally.

In [1]:
import sys, time
from pathlib import Path
import duckdb

SRC = "../../data/climate_trace/v5.11.0/emissions/DATA/emissions_co2e_100yr_v5_11_0.csv"
DST = "../../data/climate_trace/v5.11.0/emissions/emissions_co2e_100yr_v5_11_0.parquet"

assert Path(SRC).exists(), f"Missing source CSV: {Path(SRC).resolve()}"
print(sys.executable, sys.version.split()[0], duckdb.__version__)

/opt/miniconda3/envs/carboncap/bin/python 3.12.14 1.5.6


## 1. Converting the CSV to Parquet

My first attempt at the conversion failed. DuckDB guesses column types from the first 20,480 rows, decided `other8` was a number, and then hit the text value `"hp"` on line 3,922,262. The `otherN` columns mean something different in each subsector, so no single type fits them.

I switched to `all_varchar=true` and kept every column as text. That is how I want bronze to work anyway: store the source exactly as it arrives, then cast in silver with `TRY_CAST` and count whatever fails, instead of letting one bad value stop the load.

The conversion took 9 seconds and shrank 9.3 GB of CSV to a 160 MB Parquet file with zstd compression. Parquet stores data by column and each query reads only the columns it needs, which is why every query below runs in under a second.

In [2]:
# Only convert if the Parquet file isn't there yet
if not Path(DST).exists():
    t0 = time.time()
    duckdb.sql(f"""
        COPY (SELECT * FROM read_csv('{SRC}', all_varchar=true))
        TO '{DST}' (FORMAT parquet, COMPRESSION zstd)
    """)
    print(f"Converted in {time.time() - t0:.0f}s")
else:
    print("Parquet already exists, skipping conversion")
print(f"Parquet size: {Path(DST).stat().st_size / 1e6:.0f} MB")

Parquet already exists, skipping conversion
Parquet size: 160 MB


In [3]:
duckdb.sql(f"""DESCRIBE SELECT * FROM '{DST}'""")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ source_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ source_name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ source_type   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ iso3_country  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sector        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ subsector     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_time    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ end_time      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lat           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lon           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

## 2. A typed view for exploring

The `em` view casts only the columns I need. I kept the raw text in `emissions_raw` next to the converted `emissions_t` so I can count values that fail to convert.

In [4]:
con = duckdb.connect()   # in-memory database; the data stays in the Parquet file

con.sql(f"""
    CREATE OR REPLACE VIEW em AS
    SELECT
        source_id,
        source_name,
        source_type,
        sector,
        subsector,
        TRY_CAST(start_time AS TIMESTAMP)          AS start_time,
        emissions_quantity                          AS emissions_raw,
        TRY_CAST(emissions_quantity AS DOUBLE)      AS emissions_t,
        geometry_ref
    FROM '{DST}'
""")

In [5]:
con.sql("""
    SELECT
        COUNT(*)                   AS row_count,
        COUNT(DISTINCT source_id)  AS distinct_sources,
        MIN(start_time)            AS first_month,
        MAX(start_time)            AS last_month
    FROM em
""").show()

┌───────────┬──────────────────┬─────────────────────┬─────────────────────┐
│ row_count │ distinct_sources │     first_month     │     last_month      │
│   int64   │      int64       │      timestamp      │      timestamp      │
├───────────┼──────────────────┼─────────────────────┼─────────────────────┤
│  12481380 │           208023 │ 2021-01-01 00:00:00 │ 2025-12-01 00:00:00 │
└───────────┴──────────────────┴─────────────────────┴─────────────────────┘



The data is a complete grid. There are 12,481,380 rows, which is exactly 208,023 sources times 60 months, so no source is missing a month and none has extras. This is worth checking in silver for every new release.

In [6]:
con.sql("""
    SELECT
        sector,
        subsector,
        COUNT(*)                              AS row_count,
        COUNT(DISTINCT source_id)             AS sources,
        ROUND(SUM(emissions_t) / 1e6, 2)      AS emissions_mt,     -- million tonnes CO2e, 2021-2025
        ROUND(100 * SUM(emissions_t)
              / SUM(SUM(emissions_t)) OVER (), 2) AS pct_of_total
    FROM em
    GROUP BY sector, subsector
    ORDER BY emissions_mt DESC NULLS LAST
""").show(max_rows=70)

┌────────────────────────┬───────────────────────────────────────────────┬───────────┬─────────┬──────────────┬──────────────┐
│         sector         │                   subsector                   │ row_count │ sources │ emissions_mt │ pct_of_total │
│        varchar         │                    varchar                    │   int64   │  int64  │    double    │    double    │
├────────────────────────┼───────────────────────────────────────────────┼───────────┼─────────┼──────────────┼──────────────┤
│ transportation         │ road-transportation                           │    204480 │    3408 │     12647.08 │        31.93 │
│ power                  │ electricity-generation                        │    151680 │    2528 │      8017.49 │        20.24 │
│ buildings              │ residential-onsite-fuel-usage                 │    204480 │    3408 │      3090.16 │          7.8 │
│ fossil-fuel-operations │ oil-and-gas-production                        │     34320 │     572 │       2399.3 │

### Where the emissions are

* Only 46 of the 66 subsectors I asked for have any US data.
* About 55% of emissions come from subsectors that Climate TRACE reports by area rather than by facility, like road transport (32%), buildings (12%), forestry and farmland. None of these have owners.
* Cattle operations and food manufacturing make up about 60% of the rows but only about 3% of emissions.
* The subsectors this project cares about (power, oil and gas, coal, chemicals, steel and metals, cement, lime and glass) are about 37% of emissions and less than 5% of the rows.
* Oil and gas production and oil and gas transport both have exactly 572 sources. That makes me think they are basins or fields rather than individual facilities. I will check this when I look at ownership.
* Forest land degradation adds up to zero.

In [7]:
con.sql("""
    SELECT
        COALESCE(NULLIF(TRIM(source_type), ''), '<empty>') AS source_type,
        COUNT(*)                                            AS row_count,
        COUNT(DISTINCT source_id)                           AS sources,
        COUNT(DISTINCT subsector)                           AS subsectors,
        ANY_VALUE(subsector)                                AS example_subsector
    FROM em
    GROUP BY 1
    ORDER BY row_count DESC
""").show(max_rows=100)

┌─────────────────────────────────────────┬───────────┬─────────┬────────────┬───────────────────────────────────────────────┐
│               source_type               │ row_count │ sources │ subsectors │               example_subsector               │
│                 varchar                 │   int64   │  int64  │   int64    │                    varchar                    │
├─────────────────────────────────────────┼───────────┼─────────┼────────────┼───────────────────────────────────────────────┤
│ <empty>                                 │   3433320 │   57222 │         21 │ synthetic-fertilizer-application              │
│ enteric_fermentation_otherbeefcattle    │   1579800 │   26330 │          1 │ enteric-fermentation-cattle-operation         │
│ otherbeefcattle                         │   1579800 │   26330 │          1 │ manure-management-cattle-operation            │
│ maturedairycattle                       │   1320060 │   22001 │          1 │ manure-management-cattle-operati

`source_type` is free text, and what it describes changes from one subsector to the next: cattle type, treatment level, fuel mix and so on. For power plants it lists fuels in no fixed order, so "gas, oil" and "oil, gas" show up as different values. Silver will need to split and sort these before grouping. It is empty for 21 subsectors, so I can't use it to tell facilities from areas.

In [8]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE emissions_raw IS NULL OR TRIM(emissions_raw) = '')     AS empty_values,
        COUNT(*) FILTER (WHERE NULLIF(TRIM(emissions_raw), '') IS NOT NULL
                           AND emissions_t IS NULL)                                AS failed_casts,
        COUNT(*) FILTER (WHERE emissions_t < 0)                                    AS negative_values
    FROM em
""").show()

# If failed_casts > 0, look at what they contain:
con.sql("""
    SELECT emissions_raw, COUNT(*) AS n
    FROM em
    WHERE NULLIF(TRIM(emissions_raw), '') IS NOT NULL AND emissions_t IS NULL
    GROUP BY 1 ORDER BY n DESC LIMIT 20
""").show()

┌──────────────┬──────────────┬─────────────────┐
│ empty_values │ failed_casts │ negative_values │
│    int64     │    int64     │      int64      │
├──────────────┼──────────────┼─────────────────┤
│            0 │            0 │              24 │
└──────────────┴──────────────┴─────────────────┘

┌───────────────┬───────┐
│ emissions_raw │   n   │
│    varchar    │ int64 │
└───────────────┴───────┘
         0 rows        



The emissions column is clean. There are no empty values and no values that fail to convert to a number. There are 24 negative values, which I look at in section 5.

While reading the raw CSV I also noticed two oddities. `other4` sometimes holds a Python object printed as text (`{'ss': np.float64(...)}`) instead of proper JSON. And in manure management, the `otherN` values don't always seem to match their `otherN_def` labels. I haven't confirmed the second one yet.

## 3. Facilities vs areas

`geometry_ref` tells me what kind of source a row is. It is empty for facilities, starts with `gadm_` for counties (3,148 of them) and with `ghs-fua_` for urban areas (260).

The query below is written as a test. It returns any subsector that breaks the rule "every subsector is either all facilities or all areas, and every prefix is one I know about". If the data follows the rule, it returns nothing.

In [9]:
# Returns the subsectors that break the rule. No rows means the test passed.
con.sql("""
    SELECT
        subsector,
        COUNT(DISTINCT source_id) FILTER (WHERE geometry_ref IS NULL) AS facility_sources,
        COUNT(DISTINCT source_id) FILTER (WHERE split_part(geometry_ref, '_', 1) = 'gadm') AS gadm_sources,
        COUNT(DISTINCT source_id) FILTER (WHERE split_part(geometry_ref, '_', 1) = 'ghs-fua') AS fua_sources,
        COUNT(DISTINCT source_id) FILTER (WHERE geometry_ref IS NOT NULL) AS area_sources,
        COUNT(DISTINCT source_id) AS total_sources
    FROM em
    GROUP BY subsector
    HAVING (facility_sources > 0)::INT + (area_sources > 0)::INT > 1
    OR total_sources <> facility_sources + gadm_sources + fua_sources
    ORDER BY subsector
    
""").show(max_rows=50)

┌───────────┬──────────────────┬──────────────┬─────────────┬──────────────┬───────────────┐
│ subsector │ facility_sources │ gadm_sources │ fua_sources │ area_sources │ total_sources │
│  varchar  │      int64       │    int64     │    int64    │    int64     │     int64     │
└───────────┴──────────────────┴──────────────┴─────────────┴──────────────┴───────────────┘
                                           0 rows                                         



It returned 0 rows, so the rule holds. There are 30 facility subsectors and 16 area subsectors with no mixing, which means silver can filter on subsector alone.

My first version of this query compared the prefix to `'fua'` instead of `'ghs-fua'` and quietly returned zeros. The `total_sources` comparison is there to catch that kind of mistake.

## 4. Aviation: airports or airlines?

In [10]:
# Top 10 aviation sources by emissions in each subsector

con.sql("""
    SELECT
        subsector,
        source_id,
        source_name,
        ROUND(SUM(emissions_t) / 1e6, 3) AS emissions_mt_2021_2025
    FROM em
    WHERE subsector IN ('domestic-aviation', 'international-aviation')
    GROUP BY subsector, source_id, source_name
    QUALIFY ROW_NUMBER() OVER (PARTITION BY subsector ORDER BY emissions_mt_2021_2025 DESC) <= 10
    ORDER BY subsector, emissions_mt_2021_2025 DESC
""").show(max_rows=30)

┌────────────────────────┬───────────┬──────────────────────────────────────────────────┬────────────────────────┐
│       subsector        │ source_id │                   source_name                    │ emissions_mt_2021_2025 │
│        varchar         │  varchar  │                     varchar                      │         double         │
├────────────────────────┼───────────┼──────────────────────────────────────────────────┼────────────────────────┤
│ domestic-aviation      │ 3172537   │ Los Angeles International Airport                │                 26.844 │
│ domestic-aviation      │ 3170548   │ Hartsfield Jackson Atlanta International Airport │                 26.388 │
│ domestic-aviation      │ 3173366   │ Chicago O'Hare International Airport             │                  26.25 │
│ domestic-aviation      │ 3171233   │ Denver International Airport                     │                 24.312 │
│ domestic-aviation      │ 3171235   │ Dallas Fort Worth International Airport  

Aviation emissions belong to airports (LAX, Atlanta, O'Hare, Denver, JFK and so on), with separate domestic and international sources for each airport. Airlines don't appear at all, and in the US airports are usually run by city or regional authorities. This backs up ADR 0005, which leaves aviation out of the v1 universe.

## 5. Negative emissions

In [11]:
# Every source with at least one negative month

con.sql("""
    SELECT
        subsector,
        source_id,
        source_name,
        COUNT(*)                                     AS negative_months,
        MIN(start_time)::DATE                        AS first_month,
        MAX(start_time)::DATE                        AS last_month,
        ROUND(MIN(emissions_t), 2)                   AS most_negative_t,
        ROUND(SUM(emissions_t), 2)                   AS sum_negative_t
    FROM em
    WHERE emissions_t < 0
    GROUP BY ALL
    ORDER BY subsector, negative_months DESC
""").show(max_rows=30)

┌─────────────────────────┬───────────┬─────────────────┬─────────────────┬─────────────┬────────────┬─────────────────┬────────────────┐
│        subsector        │ source_id │   source_name   │ negative_months │ first_month │ last_month │ most_negative_t │ sum_negative_t │
│         varchar         │  varchar  │     varchar     │      int64      │    date     │    date    │     double      │     double     │
├─────────────────────────┼───────────┼─────────────────┼─────────────────┼─────────────┼────────────┼─────────────────┼────────────────┤
│ net-soil-organic-carbon │ 51180310  │ Cheyenne County │              12 │ 2021-01-01  │ 2021-12-01 │         -294.06 │       -3528.68 │
│ net-soil-organic-carbon │ 51180407  │ Clark County    │              12 │ 2024-01-01  │ 2024-12-01 │            -0.0 │          -0.01 │
└─────────────────────────┴───────────┴─────────────────┴─────────────────┴─────────────┴────────────┴─────────────────┴────────────────┘



All 24 negative values come from two counties in `net-soil-organic-carbon`, where soil stores more carbon than it releases, so a negative number makes sense. Cheyenne County is negative for every month of 2021 and Clark County for every month of 2024, although Clark's values are so close to zero that they look like rounding noise. Neither is in scope.

Cheyenne has exactly the same value every month, 294.06 tonnes removed, which adds up to 3,528.68 tonnes for the year. So for some subsectors the monthly figures are really an annual estimate split evenly over twelve months.

For silver, I will add a check that emissions in the subsectors we keep are never negative.

## Summary

**What this means for the pipeline**

* Bronze keeps Climate TRACE exactly as downloaded, all as text, and records the release version. Silver does the typing and the checks.
* We can probably ingest only the facility subsectors we need. I will decide once I have explored the ownership data and record it in an ADR.
* Aviation stays out of scope, as ADR 0005 says.

**Checks to build in silver**

1. Rows equal sources times months for each release.
2. No subsector mixes facilities and areas, and every `geometry_ref` prefix is known.
3. Emissions in the subsectors we keep are never negative.
4. Failed casts are counted rather than allowed to break the load.

**Still open**

* Do the county and urban area sources overlap and double count? I can check this against the US total Climate TRACE publishes.
* Are the oil and gas production and transport sources basins or facilities?
* Are the `otherN` values misaligned with their labels in manure management?
* Which subsectors we keep have real monthly figures, and which are an annual number divided by 12?

Next I will download the ownership data and explore it in notebook 02.